# The whole loop — one question, one experiment, one report

This is the tutorial. It is one continuous piece of work rather than a tour of the
subpackages: a decision arrives, the graph refuses to answer it from the data on hand, an
experiment is designed and priced, the experiment lands, the model is calibrated against
it, the follow-up is planned, and a document goes out. Every number below is produced by
the code above it, and the last section hands the whole thing to
[`axiom-dossier`](../../packages/axiom-dossier/README.md) — the add-on that writes the
report — so nothing is retyped on the way to the page.

**The situation.** A logistics operator runs 40 distribution depots in one region and 500
nationally. Maintenance hours can be spent on a depot each week; the outcome is an on-time
delivery index. Maintenance done this week still helps next week (carryover) and the tenth
hour buys less than the first (saturation). Maintenance is not free: an hour costs money,
and the schedule only earns its keep if the lift it buys is worth more than the hours.
The programme office asks:

> **Should the standing weekly maintenance schedule go from nothing to 60 hours per
> depot?**

A year of panel data exists. That is where the trouble starts.

| phase | what it decides | subpackages |
|---|---|---|
| 1. The question | what number would settle it | `core`, `estimands` |
| 2. Identification | whether the data on hand can produce that number | `identify` |
| 3. The belief we start from | what the observational fit says, and why it is wrong | `sim`, `surface` |
| 4. Design | how big, which estimator, and whether it is worth running | `design` |
| 5. Measurement | the experiment as one typed piece of evidence | `calibrate` |
| 6. Calibration | folding the experiment back into the model | `calibrate`, `surface` |
| 7. The follow-up | what to run next, and when | `design`, `estimands` |
| 8. The report | the document, written from the evidence | `axiom_dossier` |

Runtime is well under a minute: two surface fits and a small simulation study.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

from axiom.calibrate import (
    Agreement,
    Correction,
    Ledger,
    Measurement,
    ResolvedTransfer,
    aggregation_level,
    agreement,
    carryover_window_factor,
    fit_calibrated,
    resolve,
)
from axiom.core import Assumption, Intervention, LedgerLine, Population, TimeWindow, wald
from axiom.data import Panel
from axiom.design import (
    AnchoredEffect,
    DecisionSpec,
    DesignCandidate,
    EconomicInputs,
    SampleSize,
    SimulatedPower,
    SimulationSpec,
    ValuePerOutcome,
    anchor_draws,
    calibrate_registry,
    difference_in_differences_se,
    eig_gaussian,
    evaluate_candidate,
    evoi_gaussian,
    experiment_value,
    information_value_of,
    opportunity_cost,
    pareto_front,
    power_from_se,
    sample_size,
    simulated_power,
    time_to_re_experiment,
)
from axiom.estimands import Estimand, EstimandResult, Level, Quantity, RealizedDraws, TransferPlan, realize
from axiom.identify import CausalGraph, identify
from axiom.sim import DosePlan, surface_world
from axiom.surface import FitResult, GeometricCarryover, HillKernel, fit

pd.set_option("display.width", 160)

SEED = 0

MASS = 0.9
"""Every interval below is a 90 % one, and every printout says which kind."""

DOSE, NO_DOSE = 60.0, 0.0
"""The schedule under consideration, in maintenance hours per depot per week."""

MAX_LAG = 4
"""Weeks of carryover: what is done now still matters four weeks out."""

VALUE_PER_POINT = 400.0
"""USD per index point per depot-week — the programme office's conversion."""

COST_PER_HOUR = 45.0
"""USD per maintenance hour, fully loaded."""

REGION_DEPOTS, PROGRAM_DEPOTS = 40, 500

HORIZON_WEEKS = 52
"""A schedule decision commits the whole network for a year; that is what it is worth."""

OUTCOME_UNIT = "index points"

## 1. The question, written down

"Should we go to 60 hours?" is not yet a quantity. Nine facets separate it from one: which
treatment, at what dose, against what reference, for whom, over what window, read at what
level. `Estimand` carries all of them — and the reason to write down *two* estimands here
is that **the experiment cannot measure the thing the decision needs**:

* the **experiment** reads a *first-week* lift, per depot: dose one week, read the next
  observation;
* the **decision** is about the *steady-state weekly* lift across the region's 40 depots —
  what a standing schedule is worth once the carryover has filled in.

They differ on the `window` and `level` facets, and `transfer_to` says so before either
number exists. That is the point of writing both now: the gap between what will be
measured and what is wanted is named at the start, when it is still cheap.

In [ ]:
TRUTH = {"beta_a": 10.0, "alpha": 5.0, "k_a": 50.0, "s_a": 2.0, "lam_a": 0.5}

world = surface_world(
    n_units=REGION_DEPOTS,
    n_periods=52,
    treatments=("a",),
    kernels=HillKernel(reference_dose=50.0, amplitude_scale=10.0),
    carryover={"a": GeometricCarryover(max_lag=MAX_LAG)},
    doses=DosePlan(scale=50.0, spread=0.8, zero_fraction=0.05),
    intercept="shared",
    truth=TRUTH,
    noise_sd=2.0,
    seed=SEED,
)
spec = world.spec


def contrast(name: str, window: TimeWindow, level: Level, dose: float = DOSE) -> Estimand:
    """The same comparison — hours against none — read on different terms."""
    return Estimand(
        name=name,
        quantity=Quantity(kind="contrast"),
        treatment=spec.treatment("a"),
        intervention=Intervention(doses={"a": dose}),
        reference=Intervention(doses={"a": NO_DOSE}),
        outcome=spec.outcome,
        population=Population(name="region_depots"),
        window=window,
        level=level,
        dimension=spec.outcome_dimension,
    )


FIRST_WEEK = TimeWindow(start=0, stop=1, basis="cumulative")
STEADY_STATE = TimeWindow(start=MAX_LAG, stop=world.n_periods, basis="per_period")

experiment_estimand = contrast("first_week_lift", FIRST_WEEK, Level(unit="individual"))
decision_estimand = contrast("steady_state_weekly_lift", STEADY_STATE, Level(unit="cluster"))
for e in (experiment_estimand, decision_estimand):
    print(f"{e.name:26s} window={e.window.start:2d}..{e.window.stop} ({e.window.basis:10s}) "
          f"level={e.level.unit:10s} hash={e.content_hash()[:12]}")

plan: TransferPlan = experiment_estimand.transfer_to(decision_estimand)
print("\nwhat the experiment reads is not what the decision needs:")
print(f"  transfer status: {plan.status} | facets that differ: {plan.differing}")

### What "yes" would have to beat

The schedule costs `60 hours x 45 USD = 2,700` per depot per week, and a point of index is
worth 400 per depot-week. So it pays for itself only if the steady-state weekly lift clears
**6.75 points per depot**. That number is not a statistical threshold — it is arithmetic on
prices — and it is what turns an estimate into a decision.

In [ ]:
WEEKLY_COST = DOSE * COST_PER_HOUR
BREAK_EVEN = WEEKLY_COST / VALUE_PER_POINT
print(f"{DOSE:.0f} hours at {COST_PER_HOUR:.0f} USD = {WEEKLY_COST:,.0f} USD per depot-week")
print(f"break-even steady-state lift: {BREAK_EVEN:.2f} {OUTCOME_UNIT} per depot-week")
print(f"across the region: {BREAK_EVEN * REGION_DEPOTS:.1f} {OUTCOME_UNIT} per week")

## 2. Can the data we already have answer it?

The panel exists because depots were maintained — not at random. The depots with newer
fleets got more scheduled hours *and* ran better, and fleet condition was never recorded.
That is a back-door path through an unmeasured node, and `identify` says so rather than
returning a coefficient.

In [ ]:
observational = CausalGraph.from_edges(
    "u -> a, u -> y, a -> y", unmeasured=("u",), name="depot-maintenance (observational)"
)
v_obs = identify(observational, "a", "y")
print(f"observational: status={v_obs.status} | route={v_obs.route} | "
      f"needs unmeasured={v_obs.unmeasured_required}")
for a in v_obs.verdict.assumptions:
    print(f"  assumption {a.name:34s} state={a.state}")

Randomizing cuts the arrow *into* the treatment: if the hours are assigned by a coin, fleet
condition cannot explain the difference any more. The same question under a randomized
assignment is `identified`, with an empty adjustment set — which is the whole point of
randomizing, stated as a graph rather than as a habit.

In [ ]:
randomized = CausalGraph.from_edges(
    "a -> y, u -> y", unmeasured=("u",), name="depot-maintenance (randomized)"
)
v_rct = identify(randomized, "a", "y")
print(f"randomized:    status={v_rct.status} | route={v_rct.route} | "
      f"adjust for={v_rct.adjustment_set or 'nothing'}")
for a in v_rct.verdict.assumptions:
    print(f"  assumption {a.name:34s} state={a.state}")

**The decision to run an experiment has now been made by the graph rather than by
enthusiasm.** Everything from here is about running it well.

## 3. The belief we start from

The observational panel is not useless — it is the prior. Fitting the response surface to
it gives a curve whose amplitude is inflated (the fit credits maintenance for what fleet
condition was doing) but whose shape is informative, and a wrong-but-stated prior is what
makes a design calculation possible at all.

In [ ]:
rng = np.random.default_rng(123)
frame = world.panel.frame
dose = frame["a"].to_numpy(dtype=np.float64)
standardized_dose = (dose - dose.mean()) / dose.std()
fleet_condition = 0.6 * standardized_dose + 0.8 * rng.standard_normal(dose.size)
observed = Panel(frame.assign(y=frame["y"] + 0.8 * fleet_condition), world.panel.roles)
print(f"panel: {world.n_units} depots x {world.n_periods} weeks = {len(observed.frame)} rows | "
      f"corr(hours, fleet condition) = {float(np.corrcoef(dose, fleet_condition)[0, 1]):.2f}")


def moments(result: FitResult) -> dict[str, tuple[float, float]]:
    return {
        n: (float(result.posterior.flat(n).mean()), float(result.posterior.flat(n).std(ddof=1)))
        for n in ("beta_a", "k_a", "s_a", "lam_a", "alpha")
    }


def show_fit(label: str, result: FitResult) -> None:
    m = moments(result)
    print(f"{label:16s}" + "  ".join(f"{n} {mu:7.3f}+-{sd:5.3f}" for n, (mu, sd) in m.items()))


biased = fit(spec, observed, backend="laplace", draws=1000, seed=1)
print(" " * 16 + "  ".join(f"{n} truth {TRUTH[n]:<9}" for n in ("beta_a", "k_a", "s_a", "lam_a", "alpha")))
show_fit("observational", biased)
b_mean, b_sd = moments(biased)["beta_a"]
print(f"\nthe amplitude sits {(b_mean - TRUTH['beta_a']) / b_sd:.1f} posterior sd above the truth — "
      "and nothing inside the panel can say so")

`realize` turns a fit into an estimand's number. The verdict from step 2 travels with it:
because the observational graph is *not* identified, realizing it takes an explicit
`assume_identified=True` — a decision a person makes and a ledger records, never a default.
`keep_draws=True` keeps the posterior draws of the contrast, and those draws are the prior
the experiment will be designed against.

In [ ]:
prior = realize(experiment_estimand, biased, assume_identified=True, keep_draws=True, mass=MASS)
assert isinstance(prior, RealizedDraws)
contrast_draws = np.asarray(prior.draws, dtype=np.float64).ravel()
prior_mean, prior_sd = float(contrast_draws.mean()), float(contrast_draws.std())
print("status:", prior.result.status, "| ledger:", [line.kind for line in prior.result.ledger])

diff = world.forward({"a": DOSE}) - world.forward({"a": NO_DOSE})  # (unit, period)
truth_experiment = float(np.mean(diff[:, 0]))
truth_decision = float(np.sum(np.mean(diff[:, MAX_LAG:], axis=1)))
print(f"\nprior on the first-week lift: {prior_mean:.3f} +- {prior_sd:.3f} {prior.result.summary.interval}")
print(f"  the truth, which the analyst does not know:  {truth_experiment:.3f}")
print(f"  the decision estimand's truth, per region:   {truth_decision:.3f} "
      f"({truth_decision / REGION_DEPOTS:.3f} per depot, against a break-even of {BREAK_EVEN:.2f})")

The prior is inflated by more than half. Nothing in this notebook has used the truth to get
here — the analyst's position is that they have a fitted surface, a verdict that says it is
not identified, and a decision to make.

## 4. Designing the experiment

Five questions, in order. Skipping any of them is how an experiment ends up too small to
settle anything, or too expensive to be worth settling.

### 4a. What effect size should it be powered for?

The programme's conventional minimum worthwhile difference is 2.0 points. But the prior
already puts essentially all of its mass above that, and an experiment powered for a number
the model already believes buys information the model already has. `anchor_draws` reports
`P(effect > MDE)` and, when the answer is "already believed", the effect the model *doubts*
at 90 % credence. That is what to power for.

In [ ]:
MDE_CONVENTIONAL = 2.0
anchored = anchor_draws(contrast_draws, experiment_estimand.name, mde=MDE_CONVENTIONAL, credence=0.9)
assert isinstance(anchored, AnchoredEffect)
print(f"P(lift > {MDE_CONVENTIONAL}) = {anchored.probability_exceeds_mde:.3f} -> "
      f"already believed: {anchored.already_believed}")
print(f"anchored effect (the 10 % quantile the model doubts): {anchored.anchored_effect:.3f}")
effect = anchored.anchored_effect if anchored.already_believed else MDE_CONVENTIONAL
print(f"powering for {effect:.3f} {OUTCOME_UNIT}")

### 4b. How big?

One normal-model formula answers power, MDE and sample size, and `sample_size` is the exact
inversion of `power`, so the achieved power at the returned `n` is the target rather than an
approximation. The outcome's standard deviation comes from the fit's own `sigma`: a design
is not allowed to invent a noise level.

In [ ]:
sd = float(biased.posterior.summary("sigma").mean)
ss = sample_size(effect=effect, sd=sd, power=0.8, alpha=0.05)
conventional = sample_size(effect=MDE_CONVENTIONAL, sd=sd, power=0.8, alpha=0.05)
assert isinstance(ss, SampleSize) and isinstance(conventional, SampleSize)
print(f"outcome sd from the fit: {sd:.3f}")
print(f"n for the anchored effect {effect:.2f}: {ss.n} depots "
      f"({ss.n_treated} treated / {ss.n_control} held out), power {ss.power:.3f}")
print(f"n for the conventional {MDE_CONVENTIONAL:.2f}: {conventional.n} depots — "
      f"the region has {REGION_DEPOTS}, so both are runnable")

### 4c. Is the answer worth what it costs?

`eig_gaussian` prices an experiment in nats. To price it in money the *decision* has to be
stated, and here it already is: raise the schedule across all 500 depots if the steady-state
lift clears the break-even of 6.75 points. The prior lives on the experiment's scale (a
first-week lift), so the threshold has to be carried onto that scale too —
`carryover_window_factor` says what share of the carryover mass lands inside a one-week
window, and that share is the conversion.

`evoi_gaussian` then returns EVPI (what a clairvoyant would be worth) and EVSI (what *this*
experiment's answer is worth). Both scale with what the decision commits: a standing schedule
covers all 500 depots for a year, so a point of weekly lift is worth
`400 x 500 x 52` before the experiment is charged for anything. Holding depots out is not free — a held-out depot forgoes the
lift it would have had — and `opportunity_cost` prices that, signed, taking the marginal
value ratio as *draws* so its uncertainty travels. `experiment_value` nets the three.

In [ ]:
lam_prior = float(biased.posterior.flat("lam_a").mean())
share = carryover_window_factor(GeometricCarryover(max_lag=MAX_LAG), {"lam_a": lam_prior}, 1, treatment="a")
assert isinstance(share, Correction)
threshold_first_week = BREAK_EVEN * share.counterfactual
print(f"{share.counterfactual:.3f} of the carryover mass lands in the first week "
      f"(at lam_a = {lam_prior:.3f})")
print(f"break-even on the experiment's scale: {threshold_first_week:.3f} {OUTCOME_UNIT}")
print(f"the prior sits at {prior_mean:.3f} +- {prior_sd:.3f} -> "
      f"P(pays for itself) = {float((contrast_draws > threshold_first_week).mean()):.2f}")

In [ ]:
decision = DecisionSpec(
    name="raise_the_schedule",
    threshold=threshold_first_week,
    value_per_outcome_unit=VALUE_PER_POINT * PROGRAM_DEPOTS * HORIZON_WEEKS,
    numeraire="USD",
)
vpo = ValuePerOutcome(
    value=VALUE_PER_POINT, outcome_unit="index point", numeraire="USD",
    source="stated by the programme office, 2026",
)
print(vpo.ledger_line().statement)

minimum_panel = SimulationSpec(
    n_units=ss.n, n_periods=20, n_pre=10, n_treated=ss.n_treated, noise_sd=sd,
    n_simulations=30, seed=SEED,
)
se_minimum = difference_in_differences_se(minimum_panel)
ev = evoi_gaussian(decision, prior_mean, prior_sd, se_minimum)
print(f"\nthe minimum design ({ss.n} depots, {minimum_panel.n_periods} weeks) would achieve "
      f"se {se_minimum:.3f}")
print(f"EIG {eig_gaussian(prior_sd, se_minimum):.3f} nats | EVPI {ev.evpi:,.0f} {ev.numeraire} | "
      f"EVSI {ev.evsi:,.0f} {ev.numeraire} | preposterior sd {ev.preposterior_sd:.3f}")

# Index points per maintenance hour, with its uncertainty — on the *steady-state* scale,
# because that is what an hour of maintenance is worth once the carryover has filled in.
# Dividing the first-week draws by the same share that converted the threshold keeps the
# ratio and the threshold on one scale.
ratio_draws = contrast_draws / share.counterfactual / DOSE
oc = opportunity_cost(
    ss.n_control / ss.n,
    minimum_panel.n_periods - minimum_panel.n_pre,
    dose_per_period=DOSE,
    marginal_value_ratio=ratio_draws,
    value_per_outcome=vpo,
    discount_rate=0.01,
    dose_unit="USD",
    dose_cost_per_unit=COST_PER_HOUR,
)
info = information_value_of(decision, prior_mean=prior_mean, prior_sd=prior_sd, experiment_se=se_minimum)
value = experiment_value(info, oc, fixed_cost=5_000.0)
print(f"\nhours withheld {oc.dose_withheld:,.0f} -> points forgone {oc.outcome_forgone:.1f} -> "
      f"opportunity cost {oc.value:,.0f} {oc.numeraire}")
print(f"information {value.information_value:,.0f} - opportunity {value.opportunity_cost:,.0f} - "
      f"fixed {value.fixed_cost:,.0f} = net {value.net:,.0f} {value.numeraire}")

> **What EVSI cannot see.** It prices the experiment *against the prior* — and this prior
> came from a fit the graph called unidentified. A confidently wrong prior makes an
> experiment look less valuable than it is, because the value of information is measured by
> how often a different answer would change a decision the prior is already sure about. The
> reason to run this experiment is the verdict in step 2; EVSI is how to choose between
> designs, not whether to believe the model.

### 4d. Which concrete design?

Three ways to actually run it: the minimum powered holdout from 4b, a region-wide holdout
over a longer horizon, and a switchback of the minimum size (every depot is its own
control, so the standard error per depot is smaller, but dose is withheld over more weeks).
`evaluate_candidate` scores all three against the *same* decision and prior;
`pareto_front` keeps the ones nothing dominates on net value, cost and information gain.

In [ ]:
region_panel = SimulationSpec(
    n_units=REGION_DEPOTS, n_periods=24, n_pre=12, n_treated=REGION_DEPOTS // 2, noise_sd=sd,
    n_simulations=30, seed=SEED,
)
se_region = difference_in_differences_se(region_panel)
economics = EconomicInputs(
    value_per_outcome=vpo, dose_per_period=DOSE, discount_rate=0.01,
    dose_unit="USD", dose_cost_per_unit=COST_PER_HOUR, marginal_value_ratio=float(ratio_draws.mean()),
)
candidates = [
    DesignCandidate(name="minimum_holdout", method="difference_in_differences", n_units=ss.n,
                    n_periods=minimum_panel.n_periods, holdout_fraction=ss.n_control / ss.n,
                    experiment_se=se_minimum, cost=5_000.0, cooldown_periods=2),
    DesignCandidate(name="region_wide", method="difference_in_differences", n_units=REGION_DEPOTS,
                    n_periods=region_panel.n_periods, holdout_fraction=0.5,
                    experiment_se=se_region, cost=20_000.0, cooldown_periods=2),
    DesignCandidate(name="switchback", method="switchback", n_units=ss.n, n_periods=24,
                    holdout_fraction=0.5, experiment_se=0.8 * se_minimum, cost=12_000.0,
                    cooldown_periods=1),
]
scores = [
    evaluate_candidate(c, decision, prior_mean=prior_mean, prior_sd=prior_sd,
                       economics=economics, effect=effect)
    for c in candidates
]
for s in scores:
    print(f"{s.name:16s} se={s.candidate.experiment_se:.3f} eig={s.eig:.3f} evsi={s.evsi:11,.0f} "
          f"oc={s.opportunity_cost:9,.0f} cost={s.cost:8,.0f} net={s.net_value:11,.0f} "
          f"power={s.power:.3f}")
front = pareto_front(scores, objectives=("net_value", "-cost", "eig"))
print("\nnot dominated:", [s.name for s in front])
chosen = max(front, key=lambda s: s.net_value)
print(f"chosen: {chosen.name} — {chosen.candidate.n_units} depots, "
      f"{chosen.candidate.n_periods} weeks, se {chosen.candidate.experiment_se:.3f}")

### 4e. Is the estimator trustworthy on a panel of that shape?

Six estimators in `METHODS` can read a panel. Before trusting one on *this* shape,
calibrate it under the null: `calibrate_registry` runs every method on data with no effect
and checks its false-positive count against the exact binomial region. A method that fails
comes back marked `experimental` in the returned registry, and is not a candidate.
Then `simulated_power` checks the realized power against the formula's prediction, and
reports bias and interval coverage while it is there. It is checked at the distance from
the prior to the threshold rather than at the anchored effect: at this standard error the
effect itself is never in doubt, and what the design has to do is separate "pays for
itself" from "does not".

In [ ]:
chosen_panel = region_panel if chosen.candidate.n_units == REGION_DEPOTS else minimum_panel
calibrated_methods, calibration = calibrate_registry(chosen_panel, alpha=0.05)
for r in calibration:
    print(f"{r.method:28s} fp={r.false_positive_count:2d}/{r.n_evaluated:2d} "
          f"region=[{r.region.lower}, {r.region.upper}] passed={r.passed!s:5} "
          f"-> {calibrated_methods[r.method].status}")

In [ ]:
CHOSEN_METHOD = chosen.candidate.method
se_experiment = chosen.candidate.experiment_se
gap = prior_mean - threshold_first_week
print(f"power at the anchored effect {effect:.2f}: "
      f"{power_from_se(effect, se_experiment).power:.3f} — power for the effect itself is not the "
      "binding question here")
print(f"the binding question is the distance to the threshold: {prior_mean:.2f} - "
      f"{threshold_first_week:.2f} = {gap:.2f}")

predicted = power_from_se(gap, se_experiment).power
realized_power = simulated_power(
    CHOSEN_METHOD, chosen_panel.model_copy(update={"effect": gap}), predicted_power=predicted
)
assert isinstance(realized_power, SimulatedPower)
print(f"\n{CHOSEN_METHOD} at an effect of {gap:.2f}: predicted power {predicted:.3f} | "
      f"realized {realized_power.power:.3f} ({realized_power.rejections}/{realized_power.n_evaluated})")
print(f"  bias {realized_power.bias:+.3f} | coverage {realized_power.coverage:.2f} | "
      f"within prediction: {realized_power.within_prediction}")
print(f"  status in the calibrated registry: {calibrated_methods[CHOSEN_METHOD].status}")

## 5. The experiment lands

Half the depots on the schedule, half held out, difference in differences over the pre and
post windows. What comes back is one number, one standard error, and — the part that
matters — *the estimand it is a number of*. `Measurement` carries all three plus how it was
produced, and its `target` is the content hash of the estimand, so nothing downstream can
quietly apply it to a different question.

The simulated read is the truth of the experiment's estimand — computed through the same
`forward` the likelihood and the design math call — plus noise at the design's standard
error.

In [ ]:
read_noise = float(np.random.default_rng(7).standard_normal())
measurement = Measurement(
    estimand=experiment_estimand,
    estimate=truth_experiment + se_experiment * read_noise,
    se=se_experiment,
    mass=MASS,
    method=CHOSEN_METHOD,
    n_units=chosen.candidate.n_units,
    n_periods=chosen.candidate.n_periods,
    source="depot-maintenance-rct-2026Q2",
)
settles = ("above" if measurement.interval.lower > threshold_first_week
           else "below" if measurement.interval.upper < threshold_first_week else "unsettled")
print(f"read: {measurement.estimate:.3f} +- {measurement.se:.3f}  {measurement.interval}")
print(f"the truth of that estimand is {truth_experiment:.3f}, inside the interval: "
      f"{measurement.interval.lower <= truth_experiment <= measurement.interval.upper}")
print(f"target estimand {measurement.target[:12]} — the one the design was written against: "
      f"{measurement.target == experiment_estimand.content_hash()}")
print(f"\nagainst the break-even of {threshold_first_week:.3f} on this scale, the measured lift "
      f"is {settles}")

### Does the model we already had agree with it?

`agreement` realizes the experiment's estimand out of a fit and compares it with the
measurement on the measurement's own terms. This is the moment the confounding becomes
visible *from the inside* — without the experiment there was no way to see it, and with it
the gap is a number.

In [ ]:
before = agreement(biased, measurement, seed=SEED)
assert isinstance(before, Agreement)
print(f"the observational fit reads {before.posterior_mean:.3f} +- {before.posterior_sd:.3f}, "
      f"the experiment says {measurement.estimate:.3f} +- {measurement.se:.3f}")
print(f"z = {before.z:+.2f} -> verdict: {before.verdict}")

## 6. Calibration — folding the experiment into the model

Two routes, and the difference between them is worth understanding.

* **The prior route** (`derive_prior`) turns the measurement into a prior on the amplitude
  and refits. It moves one parameter, through a design factor computed under the old curve.
* **The likelihood route** (`fit_calibrated`) keeps every prior and adds a soft constraint:
  the realized contrast, evaluated through `forward`, must sit within the measurement's
  standard error of it. Because that contrast depends on the saturation and the carryover
  as well as on the amplitude, the constraint pulls the *shape* too.

The likelihood route is used here. The constraint it added is recorded in the fit's own
provenance, so the refit cannot be mistaken later for an ordinary one.

In [ ]:
calibrated = fit_calibrated(spec, observed, [measurement], backend="laplace", draws=1000, seed=1)
assert isinstance(calibrated, FitResult)
print(" " * 16 + "  ".join(f"{n} truth {TRUTH[n]:<9}" for n in ("beta_a", "k_a", "s_a", "lam_a", "alpha")))
show_fit("observational", biased)
show_fit("calibrated", calibrated)
print("\nroute:", calibrated.provenance["route"],
      "| constraint:", calibrated.provenance["constraints"][0]["name"])

after = agreement(calibrated, measurement, seed=SEED)
assert isinstance(after, Agreement)
print()
print(pd.DataFrame([
    {"fit": "observational", "reads": before.posterior_mean, "sd": before.posterior_sd,
     "z": before.z, "verdict": before.verdict, "beta_a": moments(biased)["beta_a"][0]},
    {"fit": "calibrated", "reads": after.posterior_mean, "sd": after.posterior_sd,
     "z": after.z, "verdict": after.verdict, "beta_a": moments(calibrated)["beta_a"][0]},
]).round(3).to_string(index=False))
print(f"measurement {measurement.estimate:.3f} | estimand truth {truth_experiment:.3f} | "
      f"true beta_a {TRUTH['beta_a']}")

### The decision's number, two ways

The calibrated model can realize the decision estimand directly — that is what having one
`forward` buys. A memo also wants the experiment's *own* number read on the decision's
terms, and that read is not free: the window differs (a first-week lift is not a
steady-state weekly one) and so does the level (per depot is not per region).

`transfer_to` named those facets in step 1; each now gets a `Correction` carrying the
assumption it rests on, and `resolve` combines them and writes the ledger.
`carryover_window_factor` scales a first-week read to steady state using the carryover
weights at the *calibrated* decay; `aggregation_level` reads an individual effect at
cluster level.

In [ ]:
decision_before = realize(decision_estimand, biased, assume_identified=True, mass=MASS, seed=SEED)
decision_read = realize(decision_estimand, calibrated, assume_identified=True, mass=MASS, seed=SEED)
assert isinstance(decision_before, EstimandResult) and isinstance(decision_read, EstimandResult)

lam_hat = float(calibrated.posterior.flat("lam_a").mean())
window = carryover_window_factor(GeometricCarryover(max_lag=MAX_LAG), {"lam_a": lam_hat}, 1, treatment="a")
level = aggregation_level(experiment_estimand.level, decision_estimand.level,
                          cluster_size=REGION_DEPOTS, icc=0.05)
assert isinstance(window, Correction) and isinstance(level, Correction)
resolved: ResolvedTransfer = resolve(plan, corrections=[window, level])
assert isinstance(resolved, ResolvedTransfer)
per_depot, per_depot_se = resolved.apply(measurement.estimate, measurement.se)
transferred = wald(per_depot * REGION_DEPOTS, per_depot_se * REGION_DEPOTS, MASS)
region_threshold = BREAK_EVEN * REGION_DEPOTS

print(f"window: factor {window.value:.3f} at lam_a = {lam_hat:.3f} | "
      f"level: point factor {level.detail['point_factor']}, se factor {level.value:.3f}")
print()
print(f"the decision estimand, per region (break-even {region_threshold:.1f}):")
print(f"  truth, which nobody in the story knows:  {truth_decision:8.2f}")
print(f"  observational model, before the experiment: {decision_before.summary.mean:8.2f} "
      f"{decision_before.summary.interval}")
print(f"  calibrated model through forward:        {decision_read.summary.mean:8.2f} "
      f"{decision_read.summary.interval}")
print(f"  experiment carried across the transfer:  {per_depot * REGION_DEPOTS:8.2f} {transferred}")
print(f"\nthe observational model said "
      f"{'raise it' if decision_before.summary.interval.lower > region_threshold else 'raise it, on the point estimate' if decision_before.summary.mean > region_threshold else 'do not'}; "
      f"the calibrated model says "
      f"{'raise it' if decision_read.summary.interval.lower > region_threshold else 'do not'}")

The experiment reversed the decision. Before it, the model's own reading of the decision
estimand sat above break-even; after it, both readings sit below, and the schedule does not
pay for itself. The two post-experiment readings do not agree with *each other*, though, and
the transferred one is the *more precise* of the two — which is exactly backwards, because its standard error carries the experiment's precision but none of the
window assumption's risk. That line assumes the share of the *effect* inside a window equals
the share of the *carryover weights* inside it, which is exact only when the response is
linear in accumulated dose; here the dose accumulates before it saturates. Which is why the
ledger line is `unverified` rather than absent, and why the report below prints both
readings and follows the model's.

In [ ]:
ledger: Ledger = resolved.ledger
print(ledger.to_frame()[["facet", "assumption", "state", "counterfactual", "value", "correction"]].to_string(index=False))
print()
print(ledger.summary())
print("\ncovers every differing facet:", ledger.check_complete(plan).status)

## 7. Planning the follow-up

The experiment settled 60 hours. Three questions remain, and each is answered with the
*calibrated* posterior rather than the one we started with.

1. **Would repeating it be worth anything?** `evoi_gaussian` against the new posterior.
2. **When does what we learned go stale?** `time_to_re_experiment`, at a stated drift rate.
3. **What should the next experiment test instead?** The surface saturates, so a *smaller*
   dose has a lower bar to clear — break-even scales linearly with the hours while the lift
   does not — and the calibrated model can be asked about doses nobody ran.

In [ ]:
posterior = realize(experiment_estimand, calibrated, assume_identified=True, keep_draws=True, mass=MASS)
assert isinstance(posterior, RealizedDraws)
posterior_draws = np.asarray(posterior.draws, dtype=np.float64).ravel()
posterior_mean, posterior_sd = float(posterior_draws.mean()), float(posterior_draws.std())
repeat = evoi_gaussian(decision, posterior_mean, posterior_sd, se_experiment)
print(f"prior into the experiment:  {prior_mean:6.3f} +- {prior_sd:.3f}")
print(f"posterior after it:         {posterior_mean:6.3f} +- {posterior_sd:.3f}")
print(f"\nrepeating the same experiment: EIG {eig_gaussian(posterior_sd, se_experiment):.3f} nats, "
      f"EVSI {repeat.evsi:,.0f} {repeat.numeraire}")
print("nats are not money: the decision is no longer close to its threshold, so a sharper "
      "answer to the same question changes nothing about what gets done.")

In [ ]:
MIN_EIG, HALF_LIFE = 1.0, 26.0
timing = time_to_re_experiment(
    posterior_sd=posterior_sd, half_life_periods=HALF_LIFE, experiment_se=se_experiment,
    min_eig=MIN_EIG, design_kind=CHOSEN_METHOD,
)
print(f"at a {HALF_LIFE:.0f}-week information half-life, a repeat is worth {MIN_EIG:.1f} nats again "
      f"after {timing.periods:.0f} weeks")
print(f"  (today it would gain {timing.eig_now:.3f} nats; the posterior sd has to drift out to "
      f"{timing.sd_at_threshold:.3f} from {timing.posterior_sd:.3f})")

In [ ]:
rows = []
for candidate_dose in (15.0, 30.0, 45.0, 60.0):
    e = contrast(f"steady_state_at_{candidate_dose:.0f}", STEADY_STATE, Level(unit="cluster"), dose=candidate_dose)
    r = realize(e, calibrated, assume_identified=True, mass=MASS, seed=SEED)
    assert isinstance(r, EstimandResult)
    break_even_here = candidate_dose * COST_PER_HOUR * REGION_DEPOTS / VALUE_PER_POINT
    rows.append({
        "hours": candidate_dose,
        "lift (region)": r.summary.mean,
        "lower": r.summary.interval.lower,
        "break-even": break_even_here,
        "surplus": r.summary.mean - break_even_here,
        "pays": r.summary.interval.lower > break_even_here,
    })
print(pd.DataFrame(rows).round(2).to_string(index=False))
best = max(rows, key=lambda r: r["surplus"])
shortfall = -float(best["surplus"]) / REGION_DEPOTS * float(share.counterfactual)
follow_up_size = sample_size(effect=shortfall, sd=sd, power=0.8, alpha=0.05)
assert isinstance(follow_up_size, SampleSize)
print(f"\nno dose on the grid clears its own bill. {best['hours']:.0f} hours comes closest and is "
      f"still {-best['surplus']:.0f} {OUTCOME_UNIT} per week short across the region.")
print(f"an experiment able to resolve that shortfall ({shortfall:.2f} on the experiment's scale) "
      f"would need {follow_up_size.n:,} depots at {follow_up_size.power:.2f} power, against the "
      f"{PROGRAM_DEPOTS} the network has.")
print("there is no follow-up worth running on this question: the answer is no at every dose tested, "
      "and the experiment that would overturn it does not fit in the programme.")

## 8. The report

Everything above is a typed result. `axiom-dossier` is the add-on that turns typed results
into a document — axiom's charter says rendering is someone else's job, and this is that
someone else. It depends on axiom; axiom does not know it exists, and none of the twelve
contract gates can see it.

The rule it enforces is worth stating plainly: **a number that is not in the evidence record
cannot appear in the document.** The methods section is generated from the identification
verdict and the assumptions, so it cannot describe a route nobody took; the limitations
section is generated from the assumptions still unresolved, so it cannot come out shorter
than the truth. No language model is involved in any of that.

In [ ]:
from axiom_dossier import EvidenceBuilder, build, findings_plot

RANDOM_ASSIGNMENT = Assumption(
    name="assignment_is_random",
    facet="design",
    statement="depots were assigned to the schedule by a coin, not by fleet condition",
    challenged_by="an imbalance in a pre-assignment covariate beyond chance",
    state="satisfied",
)
LINEAR_WINDOW = Assumption(
    name="window_share_is_linear_in_accumulated_dose",
    facet="window",
    statement=(
        "the share of the effect inside the first week equals the share of the carryover "
        "weights inside it"
    ),
    challenged_by="saturation applied after accumulation, which is what this surface does",
    state="unverified",
)
REGION_TRANSPORTS = Assumption(
    name="the_region_stands_for_the_network",
    facet="population",
    statement="the 500 depots nationally respond as the 40 in this region do",
    challenged_by="a second region's experiment, or a moderator analysis across depots",
    state="unverified",
)
NO_DRIFT = Assumption(
    name="the_world_holds_still_until_the_decision",
    facet="time",
    statement=(
        "fleet mix and route structure are the same when the schedule is set as when it was "
        "measured"
    ),
    challenged_by=f"the {HALF_LIFE:.0f}-week information half-life the follow-up is priced against",
    state="unverified",
)

builder = EvidenceBuilder(
    "Depot maintenance schedule",
    f"Should the standing weekly maintenance schedule go from nothing to {DOSE:.0f} hours per depot?",
)
builder.verdict(v_rct)

builder.step(
    "question",
    "The question, as an estimand",
    what=(
        f"The decision is the steady-state weekly lift of {DOSE:.0f} maintenance hours against none, "
        f"read across the region over weeks {MAX_LAG}-{world.n_periods}. The experiment reads the "
        "first-week lift per depot. The two differ on the window and level facets, and the "
        f"schedule pays for itself only above {BREAK_EVEN:.2f} {OUTCOME_UNIT} per depot-week "
        f"({region_threshold:.0f} across the region)."
    ),
    why=(
        "Writing both estimands down before the experiment is what lets the gap between them be "
        "priced afterwards instead of assumed away."
    ),
    detail={
        "decision estimand": decision_estimand.name,
        "experiment estimand": experiment_estimand.name,
        "differing facets": ", ".join(plan.differing),
        "break-even (per depot-week)": f"{BREAK_EVEN:.2f}",
        "interval mass": f"{MASS:.0%}",
    },
)
builder.step(
    "identification",
    "Why an experiment at all",
    what=(
        "The observational panel has an unmeasured common cause of maintenance hours and on-time "
        f"delivery, so the effect there is {v_obs.status} — the only route needs a node nobody "
        f"recorded. Randomized assignment cuts that arrow and the effect is {v_rct.status} with an "
        "empty adjustment set."
    ),
    why=(
        "The graph, not the analyst's confidence in the model, is what decided that an experiment "
        "was needed."
    ),
    detail={
        "observational status": v_obs.status,
        "observational route": str(v_obs.route),
        "randomized status": v_rct.status,
        "adjustment set": ", ".join(v_rct.adjustment_set) or "empty",
    },
    assumptions=[RANDOM_ASSIGNMENT],
)
builder.step(
    "design",
    "Design",
    what=(
        f"The experiment was powered for {effect:.2f} {OUTCOME_UNIT} — the effect the prior doubted "
        f"at 90 % credence, rather than the conventional {MDE_CONVENTIONAL:.1f} it already believed. "
        f"Three designs were scored against the same decision and prior; the one run was "
        f"{chosen.name.replace('_', ' ')}: {chosen.candidate.n_units} depots over "
        f"{chosen.candidate.n_periods} weeks, half held out, standard error {se_experiment:.3f}. Six "
        "estimators were calibrated under the null on a panel of that shape before one was chosen."
    ),
    why=(
        "An experiment powered for an effect the model already believes buys information it already "
        "has, and an estimator not calibrated on this panel shape has an unknown false positive rate "
        "on it."
    ),
    detail={
        "powered for": f"{effect:.2f}",
        "depots": str(chosen.candidate.n_units),
        "weeks": str(chosen.candidate.n_periods),
        "estimator": CHOSEN_METHOD,
        "achieved se": f"{se_experiment:.3f}",
        "designs compared": ", ".join(s.name for s in scores),
    },
)
builder.step(
    "calibration",
    "Calibration",
    what=(
        "The observational surface fit disagreed with the experiment "
        f"(z = {before.z:+.2f}, {before.verdict}). Refitting with the measurement as a likelihood "
        f"constraint on the realized contrast moved the fit to {after.verdict} (z = {after.z:+.2f}) "
        f"and the amplitude from {moments(biased)['beta_a'][0]:.2f} to "
        f"{moments(calibrated)['beta_a'][0]:.2f}."
    ),
    why=(
        "The constraint runs through the same forward the likelihood uses, so it pulls the curve's "
        "shape and not only its height."
    ),
    detail={
        "route": str(calibrated.provenance["route"]),
        "constraint": str(calibrated.provenance["constraints"][0]["name"]),
        "amplitude before": f"{moments(biased)['beta_a'][0]:.2f}",
        "amplitude after": f"{moments(calibrated)['beta_a'][0]:.2f}",
    },
)
builder.step(
    "transfer",
    "Reading the experiment on the decision's terms",
    what=(
        "The measured first-week per-depot lift was carried to the decision's steady-state "
        f"per-region estimand by a window factor of {window.value:.3f} and a level correction over "
        f"{REGION_DEPOTS} depots, giving {per_depot * REGION_DEPOTS:.1f} +- "
        f"{per_depot_se * REGION_DEPOTS:.1f}. The calibrated model's own reading of the same estimand "
        f"is {decision_read.summary.mean:.1f}."
    ),
    why=(
        "The experiment measured what an experiment can measure; the decision needs a different "
        "window and a different level, and each step is licensed separately or not at all."
    ),
    detail={
        "window factor": f"{window.value:.3f}",
        "level se factor": f"{level.value:.3f}",
        "ledger lines": str(len(ledger.lines)),
        "ledger complete": ledger.check_complete(plan).status,
    },
    assumptions=[LINEAR_WINDOW],
)
builder.step(
    "follow_up",
    "The follow-up",
    what=(
        f"Repeating the same experiment would gain {eig_gaussian(posterior_sd, se_experiment):.2f} "
        f"nats and {repeat.evsi:,.0f} USD; a repeat is worth {MIN_EIG:.1f} nats again only after about "
        f"{timing.periods:.0f} weeks of drift at a {HALF_LIFE:.0f}-week information half-life. Read "
        f"against the calibrated surface, no dose between {rows[0]['hours']:.0f} and {DOSE:.0f} hours "
        f"clears its own bill; {best['hours']:.0f} hours comes closest and is still "
        f"{-best['surplus']:.0f} {OUTCOME_UNIT} per week short across the region. An experiment able "
        f"to resolve that shortfall would need {follow_up_size.n:,} depots against the "
        f"{PROGRAM_DEPOTS} the network has."
    ),
    why=(
        "Information gain and decision value are different quantities: once the decision is no longer "
        "near its threshold, a sharper answer to the same question is worth nats and no money. And a "
        "follow-up that does not fit in the programme is a result, not a plan."
    ),
    detail={
        "repeat EVSI (USD)": f"{repeat.evsi:,.0f}",
        "weeks until a repeat is worth 1 nat": f"{timing.periods:.0f}",
        "best dose on the grid (hours)": f"{best['hours']:.0f}",
        "shortfall there (region, per week)": f"{-best['surplus']:.0f}",
        "depots a follow-up would need": f"{follow_up_size.n:,}",
    },
    assumptions=[NO_DRIFT],
)

builder.finding(
    "experiment", measurement,
    label=f"First-week lift, {DOSE:.0f} hours vs none (measured)",
    unit=OUTCOME_UNIT, threshold=threshold_first_week, beneficial="higher",
    source=f"{CHOSEN_METHOD} on the randomized panel",
    note=(
        "The threshold is the break-even carried onto the experiment's window by the carryover "
        "share, so the comparison is like for like."
    ),
)
builder.finding(
    "decision_model", decision_read,
    label="Steady-state weekly lift, region (calibrated model)",
    unit=OUTCOME_UNIT, threshold=region_threshold, beneficial="higher",
    source="surface.fit_calibrated -> estimands.realize",
    note=(
        "The decision's own estimand, read through the one forward. Wider than the transferred "
        "number, and the width is the honest part."
    ),
)
builder.finding(
    "decision_transfer", transferred,
    label="Steady-state weekly lift, region (experiment carried across)",
    unit=OUTCOME_UNIT, threshold=region_threshold, beneficial="higher",
    source="calibrate.resolve applied to the measurement",
    note=(
        "Precise because it inherits the experiment's standard error, and no more trustworthy for "
        "it: the window correction it passed through is unverified."
    ),
)
builder.diagnostic("agreement_before", before.z, label="Agreement z, observational fit", precision=2)
builder.diagnostic("agreement_after", after.z, label="Agreement z, calibrated fit", precision=2)
builder.diagnostic("realized_power", realized_power.power,
                   label="Realized power at the distance to the threshold", precision=3)
builder.diagnostic("coverage", realized_power.coverage, label="Interval coverage in the design simulation", precision=2)
builder.diagnostic("depots", float(chosen.candidate.n_units), label="Depots randomized", precision=0)
builder.diagnostic("weeks", float(chosen.candidate.n_periods), label="Weeks observed", precision=0)

builder.assume(RANDOM_ASSIGNMENT, LINEAR_WINDOW, REGION_TRANSPORTS, NO_DRIFT)
builder.ledger_lines([
    LedgerLine(kind=line.kind, statement=line.statement, assumption=line.assumption)
    for line in ledger.lines
])
builder.remark(
    "The observational model put the schedule above break-even on its point estimate; after the "
    "experiment, every reading of the decision estimand sits below it. That reversal is the whole "
    "return on running the experiment."
)
builder.remark(
    "The transferred reading and the model's own reading disagree, and the transferred one is the "
    "more precise. That is the wrong way round: its standard error carries the experiment's "
    "precision and none of the window assumption's risk. The recommendation follows the model."
)
builder.provenance(
    seed=str(SEED),
    world="axiom.sim.surface_world (depot maintenance)",
    surface_spec=spec.content_hash()[:16],
    decision_estimand=decision_estimand.content_hash()[:16],
    measurement=measurement.content_hash()[:16],
    interval_mass=f"{MASS:.0%}",
)
evidence = builder.build()
print("evidence hash:", evidence.content_hash())
print("unresolved assumptions:", ", ".join(evidence.unresolved()) or "none")
print()
for q in evidence.findings:
    print(f"  {q.label:58s} {q.stated():44s} [{q.against_threshold()}]")

`findings_plot` draws exactly what the record holds: one row per finding, its interval, and
the threshold the decision turns on. Three colours, not two — a row whose interval *spans*
the threshold is grey, because unsettled is a third state and drawing it as either of the
other two is the lie this package exists to avoid.

In [ ]:
findings_plot(evidence)

`build` assembles the document. `style="journal"` gives it the shape a reader navigates by
habit — abstract, introduction, methods, results, model checking, discussion, conclusions,
limitations, provenance — and `verbosity="full"` puts the per-step detail, the assumption
recaps and the ledger in it. HTML keeps the figures as live plotly charts; PDF and PPTX
rasterise them.

A `Narrator` can be passed to rewrite the prose through a language model. Every numeral in
the rewrite is then checked back against the evidence, and a section that invents one keeps
its generated draft — `Dossier.rejected()` says which. That path needs an API key, so it is
named here and not run.

In [ ]:
built = build(evidence, style="journal", verbosity="full")
print(built.summary())
print("sections:", ", ".join(s.title for s in built.report.sections))
print("context keys the template still needs:", built.missing() or "none")

out = Path(tempfile.mkdtemp()) / "depot-maintenance.html"
written = built.write(str(out), inline_plotly=False)
print("\nwritten:", written, f"({Path(str(written)).stat().st_size / 1024:.0f} kB)")

## What this tutorial decided

1. **The question became an estimand** — two of them, because the experiment cannot read the
   one the decision needs, plus a break-even that turns an estimate into a decision.
2. **The graph refused the observational answer.** The experiment was run because `identify`
   said the panel could not settle it, not because someone wanted an experiment.
3. **It was powered for the effect the model doubted**, sized from the fit's own noise level,
   and read with an estimator calibrated on a panel of that shape.
4. **It was priced before it was run** — EVSI against the opportunity cost of holding depots
   out, across three concrete designs, with EVSI's blind spot stated rather than hidden.
5. **The model disagreed with the experiment, and the disagreement was a number**, which the
   likelihood route then closed.
6. **The transfer to the decision's estimand was ledgered**, and that ledger is why the more
   precise number is not the one the recommendation follows.
7. **The follow-up was priced rather than assumed.** Repeating the experiment is worth nats
   and no money, no dose on the grid pays for itself, and the experiment that would overturn
   that needs more depots than the network has — so the honest plan is to stop.
8. **The report was written from the evidence record**, so every number in it traces to a
   result above, and the limitations section could not come out shorter than the truth.

Where to go next: `nbs/end-to-end/` takes each of these phases further on its own,
`nbs/case-studies/` works three full studies (a dose-finding trial, a tutoring programme, a
physics experiment), and each subpackage's series under `nbs/` shows every public symbol it
exports.